# PURPOSE 

SpLR code used 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpLR_ND(nn.Module):
    """
    SpLR activation for (..., F) tensors (e.g., Transformer FFN hidden states).

    Formula:
        y = x + α * x * exp(-β * x^2)

    Golden init:
        alpha_raw = 0.182  -> alpha = 2*tanh(alpha_raw)   ≈ 0.36 at start
        beta_raw  = -1.35  -> beta  = 0.01 + softplus(.)  ≈ 0.24 at start

    Parameters:
        alpha_raw: per-feature, trainable (broadcasts over batch/seq)
                  shape = (1, 1, F)
        beta_raw : scalar, trainable, stage-shared (one per activation block)
    """
    def __init__(self, num_features: int):
        super().__init__()
        self.alpha_raw = nn.Parameter(torch.full((1, 1, num_features), 0.182))
        self.beta_raw  = nn.Parameter(torch.tensor(-1.35))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        alpha = 2.0 * torch.tanh(self.alpha_raw)   # signed amplitude (per feature)
        beta  = 0.01 + F.softplus(self.beta_raw)   # positive width (scalar)
        return x + alpha * x * torch.exp(-beta * x * x)


# 📘 Benchmark Log — T-0  
## Raw Activation Power on Hard Text Classification

**Benchmark ID:** T-0  
**Domain:** Text classification  
**Objective:** Evaluate *raw activation function capability* under identical conditions, without regularization crutches.

---

## 1. Purpose (WHY this benchmark exists)

This benchmark answers a focused question:

> Can SpLR function competitively as a general-purpose activation in Transformer-based text models, without tuning or architectural bias?

Earlier experiments suggested SpLR was weaker on text.  
This benchmark directly targets that weakness using a **hard dataset** and a **from-scratch Transformer**.

---

## 2. Benchmark Rules (Golden Protocol Compliance)

All conditions are identical across activations:

- Same architecture  
- Same dataset  
- Same tokenizer + vocabulary  
- Same optimizer (AdamW)  
- Same learning rate, batch size, training budget  
- No dropout (raw activation behavior only)  
- No pretrained embeddings  
- No per-activation tuning  
- No cherry-picked seeds  

The only variable is:

> The activation function inside the feed-forward blocks.

---

## 3. Dataset

**Name:** `yahoo_answers_topics`  
**Task:** 10-class topic classification  
**Input fields used:**  
- Question title  
- Question content  
- Best answer  

These are concatenated into a single text input.

**Why this dataset:**  
This is a noisy, ambiguous, real-world dataset. It is significantly harder than IMDB or AG News and stresses representation quality rather than shortcuts.

---

## 4. Model Architecture

Custom Transformer encoder trained from scratch:

- Token + positional embeddings  
- 4 Transformer blocks  
- 4 attention heads  
- Hidden size: 256  
- FFN width: 1024  
- Classification via `[CLS]` token  
- No dropout anywhere  

The model is intentionally small to:
- Remain activation-sensitive  
- Avoid regularization masking activation behavior  
- Remain computationally realistic

---

## 5. Activations Compared

| Activation | Role |
|------|------|
| GELU | Transformer baseline |
| ReLU | Classical baseline |
| Mish | Modern smooth activation |
| SpLR | Activation under investigation |

SpLR configuration used:

- α_raw = 0.182 → α ≈ 0.36  
- β_raw = −1.35 → β ≈ 0.24  
- α: per-feature trainable  
- β: scalar, trainable, stage-shared  
- No clipping  
- No architectural assistance  

This is the **raw mechanism**, not tuned for text.

---

## 6. Training Configuration

| Parameter | Value |
|------|------|
| Optimizer | AdamW |
| Learning rate | 3e-4 |
| Weight decay | 0.01 |
| Batch size | 64 |
| Max length | 256 |
| Dropout | 0.0 |
| Mixed precision | Enabled |
| Seeds | 1 (screening phase) |
| Max steps | 20,000 |

This run is explicitly labeled:

> **Screening Phase (compute-constrained)**  
> Used to test competitiveness before multi-seed confirmation.

---

## 7. Results (Screening Phase)

| Activation | Test Accuracy |
|------|------|
| **GELU** | **0.6927** |
| SpLR | 0.6897 |
| ReLU | 0.6889 |
| Mish | 0.6886 |

---

## 8. Factual Observations (not interpretation)

- All activations converge into a narrow band (~0.688–0.693).  
- SpLR does **not collapse** on text tasks.  
- SpLR performs within **0.3% absolute accuracy** of GELU.  
- ReLU and Mish do not outperform SpLR.  
- Differences are within expected single-seed noise.

---

## 9. Interpretation (hypotheses, not claims)

These are hypotheses, not established facts:

- The benchmark appears **capacity-limited rather than activation-limited**.  
- SpLR appears **compatible with Transformer optimization dynamics**.  
- SpLR is no longer text-weak by default.

No superiority claim is made based on this run.

---

## 10. Scientific Status

This benchmark supports the statement:

> SpLR demonstrates competitive baseline performance with standard activations on hard text classification under fair constraints.

It does **not yet support** claims of superiority.

The next scientifically valid step is:

- Multi-seed evaluation (≥3 seeds)  
- Comparison limited to SpLR vs GELU  
- Reporting mean ± std  
- Diagnostic analysis (learning curves, α/β behavior)

---

## 11. Conclusion (T-0)

This benchmark establishes:

> SpLR no longer fails on text tasks.  
> It now operates within the same performance regime as established activations under fair conditions.

This positions SpLR as a **general-capable activation candidate**, not a domain-specific artifact.


# The benchamrk it self

In [ ]:
# ============================================================
# T-0 — RAW Activation Power Benchmark (TEXT, HARD)
# Dataset: yahoo_answers_topics (10-class)
# Activations: GELU vs ReLU vs Mish vs SpLR
# Dropout: 0.0  (RAW POWER)
# Step-capped (Kaggle-safe): MAX_STEPS
# Seeds: configurable
# ============================================================

import os, time, math, random
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# pip install datasets
from datasets import load_dataset

# ----------------------------
# 0) CONFIG (EDIT ONLY HERE)
# ----------------------------
@dataclass
class CFG:
    dataset: str = "yahoo_answers_topics"

    # Sequence / vocab
    max_len: int = 256
    vocab_size: int = 40000
    min_freq: int = 2
    vocab_build_examples: int = 120000  # how many train samples used to build vocab

    # Model
    d_model: int = 256
    n_head: int = 4
    n_layer: int = 4
    d_ff: int = 1024

    # Train
    batch_size: int = 64
    lr: float = 3e-4
    weight_decay: float = 0.01
    grad_clip: float = 1.0

    # Screening cap (Kaggle-safe)
    max_steps: int = 20000     # set to None for full training
    eval_every: int = 600

    # Runs
    acts: Tuple[str, ...] = ("gelu", "relu", "mish", "SpLR")
    seeds: Tuple[int, ...] = (0,)   # screening: (0,)   final: (0,1,2)

cfg = CFG()

# ----------------------------
# 1) DEVICE + SEED
# ----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ----------------------------
# 2) TOKENIZER + VOCAB (fixed per seed)
# ----------------------------
def basic_tokenize(text: str) -> List[str]:
    text = text.lower().replace("\n", " ").replace("\t", " ")
    return text.split()

def make_text(ex: Dict) -> str:
    # Yahoo fields: question_title, question_content, best_answer
    t = ex.get("question_title", "")
    c = ex.get("question_content", "")
    a = ex.get("best_answer", "")
    return (t + " " + c + " " + a).strip()

def build_vocab(texts: List[str], vocab_size: int, min_freq: int) -> Dict[str, int]:
    from collections import Counter
    c = Counter()
    for t in texts:
        c.update(basic_tokenize(t))

    specials = ["<pad>", "<unk>", "<cls>"]
    stoi = {s: i for i, s in enumerate(specials)}

    words = [w for w, f in c.items() if f >= min_freq]
    words.sort(key=lambda w: c[w], reverse=True)
    words = words[: max(0, vocab_size - len(specials))]

    for w in words:
        stoi[w] = len(stoi)
    return stoi

def encode(text: str, stoi: Dict[str, int], max_len: int) -> List[int]:
    toks = ["<cls>"] + basic_tokenize(text)
    ids = [stoi.get(tok, stoi["<unk>"]) for tok in toks[:max_len]]
    pad_id = stoi["<pad>"]
    if len(ids) < max_len:
        ids += [pad_id] * (max_len - len(ids))
    return ids

def collate_builder(stoi: Dict[str,int], max_len: int):
    pad_id = stoi["<pad>"]
    def collate_fn(examples: List[Dict]):
        texts = [make_text(x) for x in examples]
        ids = [encode(t, stoi, max_len) for t in texts]
        # label column is "topic" in this dataset
        y = [x["topic"] for x in examples]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(y, dtype=torch.long)
    return collate_fn, pad_id

# ----------------------------
# 3) ACTIVATIONS
# ----------------------------
class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))

class SpLR_ND(nn.Module):
    """
    SpLR for (..., F) tensors (Transformer FFN hidden).
    Golden init:
      alpha_raw = 0.182 -> alpha = 2*tanh(alpha_raw) ~ 0.36
      beta_raw  = -1.35 -> beta  = 0.01 + softplus(beta_raw) ~ 0.24
    """
    def __init__(self, num_features: int):
        super().__init__()
        self.alpha_raw = nn.Parameter(torch.full((1, 1, num_features), 0.182))
        self.beta_raw  = nn.Parameter(torch.tensor(-1.35))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        alpha = 2.0 * torch.tanh(self.alpha_raw)
        beta  = 0.01 + F.softplus(self.beta_raw)
        return x + alpha * x * torch.exp(-beta * x * x)

def make_activation(name: str, d_ff: int) -> nn.Module:
    if name == "gelu": return nn.GELU()
    if name == "relu": return nn.ReLU()
    if name == "mish": return Mish()
    if name == "SpLR": return SpLR_ND(d_ff)
    raise ValueError("Unknown activation: " + name)

# ----------------------------
# 4) MODEL (NO DROPOUT)
# ----------------------------
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, d_ff: int, act: nn.Module):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_head, dropout=0.0, batch_first=True)  # no dropout
        self.ln2 = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, d_ff)
        self.act = act
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x, key_padding_mask=None):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + a
        h = self.ln2(x)
        h = self.fc1(h)
        h = self.act(h)
        h = self.fc2(h)
        x = x + h
        return x

class TinyTextTransformer(nn.Module):
    def __init__(self, vocab_size: int, n_class: int, max_len: int,
                 d_model: int, n_head: int, n_layer: int, d_ff: int, act_name: str):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        act = make_activation(act_name, d_ff)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_head, d_ff, act=act) for _ in range(n_layer)
        ])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_class)

    def forward(self, ids: torch.Tensor, pad_id: int):
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(B, T)
        x = self.tok(ids) + self.pos(pos)
        key_padding_mask = (ids == pad_id)  # True = ignore
        for blk in self.blocks:
            x = blk(x, key_padding_mask=key_padding_mask)
        x = self.ln(x)
        cls = x[:, 0, :]
        return self.head(cls)

# ----------------------------
# 5) TRAIN/EVAL
# ----------------------------
@torch.no_grad()
def evaluate(model, loader, pad_id):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for ids, y in loader:
        ids, y = ids.to(DEVICE), y.to(DEVICE)
        logits = model(ids, pad_id)
        loss = F.cross_entropy(logits, y)
        loss_sum += loss.item() * y.size(0)
        pred = logits.argmax(dim=-1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total

def train_one_run(act_name: str, seed: int) -> Dict:
    set_seed(seed)

    ds = load_dataset(cfg.dataset)
    n_class = ds["train"].features["topic"].num_classes

    # Build vocab from a fixed slice of training data (per seed)
    n_vocab = min(cfg.vocab_build_examples, len(ds["train"]))
    train_slice = ds["train"].select(range(n_vocab))
    texts_for_vocab = [make_text(ex) for ex in train_slice]
    stoi = build_vocab(texts_for_vocab, cfg.vocab_size, cfg.min_freq)

    collate_fn, pad_id = collate_builder(stoi, cfg.max_len)

    train_loader = torch.utils.data.DataLoader(
        ds["train"], batch_size=cfg.batch_size, shuffle=True,
        num_workers=2, pin_memory=(DEVICE=="cuda"), collate_fn=collate_fn
    )
    test_loader = torch.utils.data.DataLoader(
        ds["test"], batch_size=cfg.batch_size, shuffle=False,
        num_workers=2, pin_memory=(DEVICE=="cuda"), collate_fn=collate_fn
    )

    model = TinyTextTransformer(
        vocab_size=len(stoi), n_class=n_class, max_len=cfg.max_len,
        d_model=cfg.d_model, n_head=cfg.n_head, n_layer=cfg.n_layer, d_ff=cfg.d_ff,
        act_name=act_name
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))

    step = 0
    best_acc = 0.0
    t0 = time.time()

    model.train()
    while True:
        for ids, y in train_loader:
            step += 1
            if cfg.max_steps is not None and step > cfg.max_steps:
                break

            ids, y = ids.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                logits = model(ids, pad_id)
                loss = F.cross_entropy(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()

            if step % cfg.eval_every == 0:
                test_loss, test_acc = evaluate(model, test_loader, pad_id)
                best_acc = max(best_acc, test_acc)
                print(f"[{act_name}][seed={seed}] step={step} "
                      f"train_loss={loss.item():.4f} | test_acc={test_acc:.4f} best={best_acc:.4f}")

        if cfg.max_steps is not None and step > cfg.max_steps:
            break

    test_loss, test_acc = evaluate(model, test_loader, pad_id)
    dt = time.time() - t0
    params = sum(p.numel() for p in model.parameters())

    return {
        "act": act_name,
        "seed": seed,
        "test_acc": float(test_acc),
        "test_loss": float(test_loss),
        "time_sec": float(dt),
        "steps": int(step),
        "params": int(params),
        "dropout": 0.0,
        "dataset": cfg.dataset,
        "max_len": cfg.max_len,
        "vocab_size": cfg.vocab_size,
        "max_steps": cfg.max_steps,
        "lr": cfg.lr,
        "wd": cfg.weight_decay,
    }

def run_all():
    results = []
    for act in cfg.acts:
        for seed in cfg.seeds:
            print("\n" + "="*70)
            print(f"RUN: act={act} seed={seed} | max_steps={cfg.max_steps} | dropout=0.0")
            print("="*70)
            r = train_one_run(act, seed)
            results.append(r)
            print("DONE:", r)

    # Summary
    import statistics as stats
    print("\n" + "="*70)
    print("SUMMARY: test_acc mean ± std over seeds")
    print("="*70)
    for act in cfg.acts:
        xs = [r["test_acc"] for r in results if r["act"] == act]
        if xs:
            print(f"{act:>5}: {stats.mean(xs):.4f} ± {stats.pstdev(xs):.4f}  (n={len(xs)})")
    return results

results = run_all()
